In [1]:
import pandas as pd
titanic_url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(titanic_url)

In [5]:
import plotly.express as px

num_cols = df.select_dtypes('number').columns

for col in num_cols:
    fig = px.box(df, y=col, color='Survived',
                 title=f'Outliers in {col} by Survival',
                 labels={'Survived': 'Survived'})
    fig.show()

In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

num_cols = df.select_dtypes('number').columns.tolist()
n_cols = 3
n_rows = -(-len(num_cols) // n_cols)  # ceiling division

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=num_cols)

for i, col in enumerate(num_cols):
    row = i // n_cols + 1
    col_pos = i % n_cols + 1

    for survived, group in df.groupby('Survived'):
        fig.add_trace(
            go.Box(y=group[col],
                   name=f'Survived={survived}',
                   legendgroup=str(survived),
                   showlegend=(i == 0),
                   marker_color='steelblue' if survived == 1 else 'tomato'),
            row=row, col=col_pos
        )

fig.update_layout(height=300 * n_rows,
                  title_text='Outliers by Survival',
                  boxmode='group')
fig.show()

In [10]:
def flag_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: {len(outliers)} outliers | lower={lower:.2f}, upper={upper:.2f}')
    return outliers

for col in ['Age', 'Fare', 'SibSp', 'Parch']:
    flag_outliers(df, col)

Age: 11 outliers | lower=-6.69, upper=64.81
Fare: 116 outliers | lower=-26.72, upper=65.63
SibSp: 46 outliers | lower=-1.50, upper=2.50
Parch: 213 outliers | lower=0.00, upper=0.00


In [11]:
def outlier_summary(df):
    rows = []
    for col in df.select_dtypes('number').columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        n_outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
        rows.append({
            'column': col,
            'lower_bound': round(lower, 2),
            'upper_bound': round(upper, 2),
            'outlier_count': n_outliers,
            'outlier_pct': round(n_outliers / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values('outlier_count', ascending=False)

outlier_summary(df)

,column,lower_bound,upper_bound,outlier_count,outlier_pct
5,Parch,0.00,0.00,213,23.91
6,Fare,-26.72,65.63,116,13.02
4,SibSp,-1.50,2.50,46,5.16
3,Age,-6.69,64.81,11,1.23
2,Pclass,0.50,4.50,0,0.00
0,PassengerId,-444.00,1336.00,0,0.00
1,Survived,-1.50,2.50,0,0.00


In [13]:
fig = px.strip(df.melt(value_vars=['Age', 'Fare', 'SibSp', 'Parch']),
               x='variable', y='value',
               color='variable',
               title='Distribution & Outliers — All Numeric Columns')
fig.show()